# ResNet-18 Multiclass testing with CONNIE image dataset

In [1]:
%run ./../notebook_init.py

import os
import torch
import optuna
import mlflow

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch.utils.data import Subset
from itertools import product
from pathlib import Path
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from torch.utils.data import DataLoader

from core import DATA_FOLDER, RESULTS_FOLDER, MODELS_FOLDER

from scripts.connie_training_utils import ModelTraining, TransformedSubset, \
    Seed, get_test_transform, IMG_SIZE, get_train_transform,\
    resnet18_model, NPYFolderDataset

Load file paths and set the computation device to GPU if available; otherwise, use CPU, and initialize the random seed

In [2]:
#train_data = os.path.join(DATA_FOLDER, "train_data_png_full")
train_data = os.path.join(DATA_FOLDER, "train_data_npy_full")
test_data = os.path.join(DATA_FOLDER, "test_data_npy")


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
seed = Seed()

cuda:0


In [3]:
trainval_dataset = NPYFolderDataset(train_data)
trainval_set = Subset(trainval_dataset, list(range(len(trainval_dataset))))


test_dataset = NPYFolderDataset(test_data)
test_set = Subset(test_dataset, list(range(len(test_dataset))))

## Training final model

In [4]:
# trial 74
hyperparam = {
    "lr": 0.0004481479831586827,
    'wd': 0.0001716842332539542,
    "step": 24,
    "gamma": 0.4402853450898717
}

num_epochs = 60

print(f"\nBest hyperparameters: {hyperparam}")
print(f"Training epochs: {num_epochs}")

# Class mapping
class_idx_map = trainval_set.dataset.class_to_idx
class_names = trainval_set.dataset.classes
print(f"\nClasses: {class_names}")
print(f"Class index mapping: {class_idx_map}")

# Create metrics directory
metrics_dir = os.path.join(RESULTS_FOLDER, "test_metrics_resnet18_multiclass")
os.makedirs(metrics_dir, exist_ok=True)


Best hyperparameters: {'lr': 0.0004481479831586827, 'wd': 0.0001716842332539542, 'step': 24, 'gamma': 0.4402853450898717}
Training epochs: 60

Classes: ['Blob', 'Diffusion Hit', 'Electron', 'Muon', 'Others']
Class index mapping: {'Blob': 0, 'Diffusion Hit': 1, 'Electron': 2, 'Muon': 3, 'Others': 4}


In [5]:
from pathlib import Path
mlflow.set_tracking_uri(Path(DATA_FOLDER) / "mlruns")
#mlflow.set_tracking_uri(os.path.join(DATA_FOLDER, "mlruns"))

print("\n" + "="*80)
print("Final Training on Train+Val Set")
print("="*80)

model_training = ModelTraining()

# Train on full train+val dataset
final_model, metrics = model_training.train_model_final(
    device=device,
    dataset=trainval_set,
    num_epochs=num_epochs,
    current_class_idx=None,  # None for multiclass
    seed=seed,
    hyperparam=hyperparam,
    model=resnet18_model,
    is_binary=False  # Multiclass
)

print(f"\nTraining complete!")
print(f"   Final Train Accuracy: {metrics['final_train_accuracy']:.4f}")
print(f"   Final Train F1:       {metrics['final_train_f1']:.4f}")
print(f"   Training time:        {metrics['training_time']:.1f}s")



Final Training on Train+Val Set
FINAL MODEL TRAINING
Training on combined train+val set with fixed epochs

Multiclass classification: 5 classes
Total samples: 3577
  Class 0: 351 (9.8%)
  Class 1: 44 (1.2%)
  Class 2: 366 (10.2%)
  Class 3: 2596 (72.6%)
  Class 4: 220 (6.2%)

Training for 60 epochs
Hyperparameters: {'lr': 0.0004481479831586827, 'wd': 0.0001716842332539542, 'step': 24, 'gamma': 0.4402853450898717}

==================== Epoch 1/60 ====================
Loss: 0.9017 | Acc: 0.6679 | Time: 4.4s
Precision: 0.4678 | Recall: 0.6871 | F1: 0.5018

==================== Epoch 2/60 ====================
Loss: 0.5549 | Acc: 0.8040 | Time: 3.9s
Precision: 0.6422 | Recall: 0.8135 | F1: 0.7060

==================== Epoch 3/60 ====================
Loss: 0.4927 | Acc: 0.8356 | Time: 4.1s
Precision: 0.6792 | Recall: 0.8328 | F1: 0.7392

==================== Epoch 4/60 ====================
Loss: 0.4843 | Acc: 0.8381 | Time: 4.1s
Precision: 0.6727 | Recall: 0.8303 | F1: 0.7343

=============

In [6]:
# Training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs_range = range(1, len(metrics["train_acc_history"]) + 1)
axes[0, 0].plot(epochs_range, metrics["train_acc_history"], 'b-', linewidth=2)
axes[0, 0].set_title("Training Accuracy", fontsize=14)
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Accuracy")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs_range, metrics["train_loss_history"], 'r-', linewidth=2)
axes[0, 1].set_title("Training Loss", fontsize=14)
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Loss")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(epochs_range, metrics["train_f1_history"], label='F1', linewidth=2)
axes[1, 0].plot(epochs_range, metrics["train_precision_history"], label='Precision', linewidth=2)
axes[1, 0].plot(epochs_range, metrics["train_recall_history"], label='Recall', linewidth=2)
axes[1, 0].set_title("Training Metrics", fontsize=14)
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Score")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].axis('off')
axes[1, 1].text(0.1, 0.5,
               f"Final Training Results:\n\n"
               f"Accuracy:  {metrics['final_train_accuracy']:.4f}\n"
               f"Precision: {metrics['final_train_precision']:.4f}\n"
               f"Recall:    {metrics['final_train_recall']:.4f}\n"
               f"F1-score:  {metrics['final_train_f1']:.4f}\n\n"
               f"Training time: {metrics['training_time']:.1f}s\n"
               f"Total samples: {metrics['total_samples']}",
               fontsize=12, verticalalignment='center',
               family='monospace')

plt.tight_layout()
training_curves_path = os.path.join(metrics_dir, "training_curves.png")
plt.savefig(training_curves_path, dpi=300, bbox_inches='tight')
plt.close()

# Training metrics
train_metrics_df = pd.DataFrame([{
    'final_train_accuracy': metrics['final_train_accuracy'],
    'final_train_precision': metrics['final_train_precision'],
    'final_train_recall': metrics['final_train_recall'],
    'final_train_f1': metrics['final_train_f1'],
    'training_time_seconds': metrics['training_time'],
    'num_epochs': num_epochs,
    'total_samples': metrics['total_samples']
}])
train_metrics_path = os.path.join(metrics_dir, "training_metrics.csv")
train_metrics_df.to_csv(train_metrics_path, index=False)
print(f"Saved training metrics: {train_metrics_path}")

# Saving final model
model_path = os.path.join(MODELS_FOLDER, "final_resnet18_multiclass.pth")
torch.save({
    'model_state_dict': final_model.state_dict(),
    'hyperparameters': hyperparam,
    'num_classes': len(class_names),
    'class_names': class_names,
    'class_idx_map': class_idx_map,
    'training_metrics': metrics,
    'num_epochs': num_epochs
}, model_path)
print(f"Saved model: {model_path}")



Saved training metrics: C:\Users\Filipe\work\sara\CONNIE-particle-classifier\results\test_metrics_resnet18_multiclass\training_metrics.csv
Saved model: C:\Users\Filipe\work\sara\CONNIE-particle-classifier\models\final_resnet18_multiclass.pth


## Testing final model

In [7]:
print("="*80)
print("TEST SET EVALUATION - ResNet18 Final Model")
print("="*80)

print(f"\nLoading model from: {model_path}")
checkpoint = torch.load(model_path, weights_only=False)

# Extract information
class_names = checkpoint['class_names']
class_idx_map = checkpoint['class_idx_map']
num_classes = checkpoint['num_classes']
hyperparam = checkpoint['hyperparameters']
train_metrics = checkpoint['training_metrics']

print(f"Model loaded successfully")
print(f"\nModel Info:")
print(f"   Classes: {class_names}")
print(f"   Num Classes: {num_classes}")
print(f"   Training Accuracy: {train_metrics['final_train_accuracy']:.4f}")
print(f"   Training F1: {train_metrics['final_train_f1']:.4f}")

# Initialize model architecture
final_model = resnet18_model(device, num_classes)
final_model.load_state_dict(checkpoint['model_state_dict'])
final_model.eval()
print("Model architecture initialized and weights loaded")


TEST SET EVALUATION - ResNet18 Final Model

Loading model from: C:\Users\Filipe\work\sara\CONNIE-particle-classifier\models\final_resnet18_multiclass.pth
Model loaded successfully

Model Info:
   Classes: ['Blob', 'Diffusion Hit', 'Electron', 'Muon', 'Others']
   Num Classes: 5
   Training Accuracy: 0.9293
   Training F1: 0.8832
Model architecture initialized and weights loaded


In [8]:
print("\n" + "="*80)
print("PREPARING TEST SET")
print("="*80)

# Get max value for normalization from test set
all_indices = list(range(len(trainval_set)))
train_max = max(trainval_set.dataset.sample_max[i] for i in all_indices)

test_subset_transformed = TransformedSubset(test_set, get_test_transform(train_max))

# Create test dataloader
test_loader = DataLoader(
    test_subset_transformed,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"Test set prepared")
print(f"   Total samples: {len(test_subset_transformed)}")
print(f"   Batch size: 64")



PREPARING TEST SET
Test set prepared
   Total samples: 632
   Batch size: 64


In [9]:
print("\n" + "="*80)
print("RUNNING INFERENCE ON TEST SET")
print("="*80)

all_preds = []
all_labels = []
all_probas = []

print("Processing batches...")
with torch.no_grad():
    for batch_idx, (inputs, labels) in enumerate(test_loader):
        inputs = inputs.to(device)

        # Forward pass
        outputs = final_model(inputs)

        # Get probabilities and predictions
        probas = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

        # Store results
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probas.extend(probas.cpu().numpy())
        
        if (batch_idx + 1) % 10 == 0:
            print(f"  Processed {(batch_idx + 1) * 64} samples...")

# Convert to numpy arrays
preds_int = np.array(all_preds)
y_test = np.array(all_labels)
probas = np.array(all_probas)

print(f"Inference complete")
print(f"   Total predictions: {len(preds_int)}")



RUNNING INFERENCE ON TEST SET
Processing batches...
  Processed 640 samples...
Inference complete
   Total predictions: 632


In [10]:
# ===== CALCULATE METRICS =====
print("\n" + "="*80)
print("TEST SET RESULTS")
print("="*80)

test_accuracy = accuracy_score(y_test, preds_int)
print(f"\nOverall Test Accuracy: {test_accuracy:.4f}")

# Classification report
report = classification_report(
    y_test,
    preds_int,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)
report_df = pd.DataFrame(report).transpose()

print("\nClassification Report:")
print(report_df)

report_path = os.path.join(metrics_dir, "test_classification_report.csv")
report_df.to_csv(report_path)
print(f"\nSaved: {report_path}")

# ===== CONFUSION MATRIX =====
print("\n" + "="*80)
print("CONFUSION MATRIX")
print("="*80)

cm = confusion_matrix(y_test, preds_int, labels=np.arange(len(class_names)))

print("\nConfusion Matrix (raw counts):")
print(cm)

# Normalized confusion matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# ===== PLOT RAW CONFUSION MATRIX =====
fig_raw, ax_raw = plt.subplots(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax_raw
)

ax_raw.set_xlabel("Predicted", fontsize=12)
ax_raw.set_ylabel("True", fontsize=12)
ax_raw.set_title(
    f"Confusion Matrix (Counts)\nAccuracy: {test_accuracy:.4f}",
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()

cm_raw_path = os.path.join(metrics_dir, "test_confusion_matrix_counts.png")
plt.savefig(cm_raw_path, dpi=300, bbox_inches='tight')
plt.close(fig_raw)

print(f"Saved: {cm_raw_path}")


# ===== PLOT NORMALIZED CONFUSION MATRIX =====
fig_norm, ax_norm = plt.subplots(figsize=(8, 6))

sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".3f",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax_norm
)

ax_norm.set_xlabel("Predicted", fontsize=12)
ax_norm.set_ylabel("True", fontsize=12)
#ax_norm.set_title(
#    "Confusion Matrix (Normalized)",
#    fontsize=14,
#    fontweight='bold'
#)

plt.tight_layout()

cm_norm_path = os.path.join(metrics_dir, "test_confusion_matrix_normalized.png")
plt.savefig(cm_norm_path, dpi=300, bbox_inches='tight')
plt.close(fig_norm)

print(f"Saved: {cm_norm_path}")

# ===== PER-CLASS ANALYSIS =====
print("\n" + "="*80)
print("PER-CLASS PERFORMANCE")
print("="*80)

per_class_results = []
for i, class_name in enumerate(class_names):
    precision = report[class_name]['precision']
    recall = report[class_name]['recall']
    f1 = report[class_name]['f1-score']
    support = report[class_name]['support']
    
    per_class_results.append({
        'Class': class_name,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Support': int(support),
        'Correct': cm[i, i],
        'Class_Accuracy': cm_normalized[i, i]
    })
    
    print(f"\n{class_name}:")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  Support:   {int(support)}")
    print(f"  Correct:   {cm[i, i]}/{int(support)} ({cm_normalized[i, i]:.2%})")

per_class_df = pd.DataFrame(per_class_results)
per_class_path = os.path.join(metrics_dir, "per_class_results.csv")
per_class_df.to_csv(per_class_path, index=False)
print(f"\nSaved: {per_class_path}")

# ===== COMPARISON WITH TRAINING =====
print("\n" + "="*80)
print("TRAINING vs TEST COMPARISON")
print("="*80)

comparison_data = {
    'Metric': ['Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1 (macro)'],
    'Training': [
        train_metrics['final_train_accuracy'],
        train_metrics['final_train_precision'],
        train_metrics['final_train_recall'],
        train_metrics['final_train_f1']
    ],
    'Test': [
        test_accuracy,
        report['macro avg']['precision'],
        report['macro avg']['recall'],
        report['macro avg']['f1-score']
    ]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df['Gap'] = abs(comparison_df['Training'] - comparison_df['Test'])

print("\n", comparison_df.to_string(index=False))

# Overfitting check
avg_gap = comparison_df['Gap'].mean()
print(f"\nAverage Train-Test Gap: {avg_gap:.4f}")

if avg_gap < 0.03:
    print("   [OK] No overfitting detected (gap < 0.03)")
elif avg_gap < 0.07:
    print("   [OK] Acceptable generalization (gap < 0.07)")
elif avg_gap < 0.10:
    print("   [WARNING] Mild overfitting (gap < 0.10)")
else:
    print("   [ERROR] Significant overfitting detected (gap > 0.10)")

comparison_path = os.path.join(metrics_dir, "train_test_comparison.csv")
comparison_df.to_csv(comparison_path, index=False)
print(f"\nSaved: {comparison_path}")

# ===== SAVE PREDICTIONS =====
print("\n" + "="*80)
print("SAVING PREDICTIONS")
print("="*80)

predictions_df = pd.DataFrame({
    'True_Label': y_test,
    'True_Class': [class_names[i] for i in y_test],
    'Predicted_Label': preds_int,
    'Predicted_Class': [class_names[i] for i in preds_int],
    'Correct': y_test == preds_int
})

# Add probability scores for each class
for i, class_name in enumerate(class_names):
    predictions_df[f'Prob_{class_name}'] = probas[:, i]

predictions_path = os.path.join(metrics_dir, "test_predictions.csv")
predictions_df.to_csv(predictions_path, index=False)
print(f"Saved all predictions: {predictions_path}")


TEST SET RESULTS

Overall Test Accuracy: 0.9193

Classification Report:
               precision    recall  f1-score     support
Blob            0.819444  0.951613  0.880597   62.000000
Diffusion Hit   0.727273  1.000000  0.842105    8.000000
Electron        0.757576  0.769231  0.763359   65.000000
Muon            0.983982  0.938865  0.960894  458.000000
Others          0.739130  0.871795  0.800000   39.000000
accuracy        0.919304  0.919304  0.919304    0.919304
macro avg       0.805481  0.906301  0.849391  632.000000
weighted avg    0.926196  0.919304  0.921268  632.000000

Saved: C:\Users\Filipe\work\sara\CONNIE-particle-classifier\results\test_metrics_resnet18_multiclass\test_classification_report.csv

CONFUSION MATRIX

Confusion Matrix (raw counts):
[[ 59   1   1   1   0]
 [  0   8   0   0   0]
 [  1   1  50   6   7]
 [ 10   0  13 430   5]
 [  2   1   2   0  34]]
Saved: C:\Users\Filipe\work\sara\CONNIE-particle-classifier\results\test_metrics_resnet18_multiclass\test_confusion

In [11]:

# ===== FINAL SUMMARY =====
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print(f"\nAll results saved in: {metrics_dir}")
print("\nFiles created:")
print("  - test_classification_report.csv")
print("  - test_confusion_matrix.png")
print("  - per_class_results.csv")
print("  - train_test_comparison.csv")
print("  - test_predictions.csv")

print(f"\nTest Accuracy:   {test_accuracy:.4f}")
print(f"Macro F1:        {report['macro avg']['f1-score']:.4f}")
print(f"Macro Recall:    {report['macro avg']['recall']:.4f}")
print(f"Macro Precision: {report['macro avg']['precision']:.4f}")

print("\n" + "="*80)
print("TEST EVALUATION COMPLETE")
print("="*80)


FINAL SUMMARY

All results saved in: C:\Users\Filipe\work\sara\CONNIE-particle-classifier\results\test_metrics_resnet18_multiclass

Files created:
  - test_classification_report.csv
  - test_confusion_matrix.png
  - per_class_results.csv
  - train_test_comparison.csv
  - test_predictions.csv

Test Accuracy:   0.9193
Macro F1:        0.8494
Macro Recall:    0.9063
Macro Precision: 0.8055

TEST EVALUATION COMPLETE
